# Customer Segmentation System
### Machine Learning with K-Means Clustering & Supervised Customer Classification
**Author**: Machine Learning Engineering Team  
**Dataset**: Mall Customers Dataset (Kaggle)  
**Technologies**: Python, Pandas, NumPy, Matplotlib, Seaborn, Scikit-learn, Joblib, Flask

---
## Project Overview
In modern retail and marketing, treating all customers identically leads to suboptimal campaign ROI and missed revenue opportunities. **Customer Segmentation** is the process of partitioning a customer base into distinct, actionable subgroups based on shared characteristics such as earning power, spending propensities, and behavioral patterns.

This notebook implements an end-to-end Machine Learning pipeline:
1. **Dataset Ingestion & Kaggle Instructions**: Graceful loading of the Mall Customers dataset.
2. **Data Preprocessing & Cleaning**: Missing values, duplicate checks, IQR outlier detection, and encoding.
3. **Exploratory Data Analysis (EDA)**: Visualizing distributions, correlations, and spending trends.
4. **K-Means Clustering**: Applying feature standardization, Elbow Method, and Silhouette Analysis to identify 5 optimal clusters.
5. **Segment Profiling**: Dynamically characterizing each cluster with actionable business strategies.
6. **Supervised Customer Classification**: Training a Logistic Regression model to predict high-spending propensity.
7. **Model Serialization**: Saving trained models and preprocessors with Joblib for web deployment.


## 1. Technologies & Environment Setup
We import essential libraries for data manipulation, numerical computing, visualization, clustering, classification, and serialization.


In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    silhouette_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# Visual styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 100
%matplotlib inline

print("All dependencies successfully imported.")


## 2. Dataset Ingestion

The dataset used is the well-known **Mall Customers** dataset, containing demographic and shopping records for 200 mall visitors:
- **CustomerID**: Unique identifier for each shopper.
- **Gender**: Customer biological sex (Male / Female).
- **Age**: Age of the customer in years.
- **Annual Income (k$)**: Customer annual income in thousands of dollars.
- **Spending Score (1-100)**: Score assigned by the mall based on customer behavior and purchasing data.

### Kaggle Download Instructions
If you do not have the dataset locally:
1. Go to Kaggle: [Customer Segmentation Tutorial in Python](https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python)
2. Click **Download** to obtain `archive.zip`.
3. Extract `Mall_Customers.csv` into the `data/` folder relative to the project root.


In [ ]:
# Path to dataset
data_path = '../data/Mall_Customers.csv'
if not os.path.exists(data_path):
    data_path = 'data/Mall_Customers.csv'

# Load dataset
df = pd.read_csv(data_path)
print(f"Dataset successfully loaded. Total rows: {df.shape[0]}, Total columns: {df.shape[1]}")
df.head(5)


## 3. Data Inspection & Preprocessing
To prepare the dataset for machine learning models, we inspect the schema, check for missing values, identify duplicates, analyze outliers using the Interquartile Range (IQR) method, and encode categorical features.


In [ ]:
# Dataset summary and data types
print("--- Dataset Info ---")
df.info()

print("\n--- Summary Statistics of Numerical Columns ---")
df.describe().round(2)


### Checking for Missing Values & Duplicate Records


In [ ]:
# Check missing values
missing_vals = df.isnull().sum()
print("Missing values per column:\n", missing_vals)

# Check duplicates
duplicate_count = df.duplicated().sum()
print(f"\nDuplicate records found: {duplicate_count}")


### Outlier Detection using the Interquartile Range (IQR) Method
An outlier is defined as any value lying below $Q_1 - 1.5 \times \text{IQR}$ or above $Q_3 + 1.5 \times \text{IQR}$.
We detect potential outliers across numerical features (`Age`, `Annual Income (k$)`, and `Spending Score (1-100)`).


In [ ]:
numerical_cols = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
outliers_dict = {}

for col in numerical_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    col_outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    outliers_dict[col] = col_outliers
    print(f"Column '{col}': IQR={iqr:.2f} | Bounds: [{lower_bound:.2f}, {upper_bound:.2f}] | Outliers detected: {len(col_outliers)}")
    if len(col_outliers) > 0:
        print(f"  -> Outlier Customer IDs: {col_outliers['CustomerID'].tolist()}")


**Outlier Handling Rationale**:
Notice that only `Annual Income (k$)` has 2 data points exceeding the statistical upper bound ($137k for customer IDs 199 and 200). These are authentic high-net-worth mall shoppers, not measurement or entry errors. Discarding them would eliminate our highest-value luxury segment. Therefore, we preserve all valid records.


### Categorical Encoding & Feature Engineering
We encode `Gender` into binary numerical format (`Male: 1, Female: 0`).


In [ ]:
df['Gender_Encoded'] = df['Gender'].map({'Male': 1, 'Female': 0})
print("Categorical Encoding complete. Value counts:")
print(df['Gender_Encoded'].value_counts())


## 4. Exploratory Data Analysis (EDA)
EDA allows us to uncover patterns, skewness, spending habits, and demographic concentrations prior to clustering.


In [ ]:
# 1. Distribution of Key Features
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
cols_meta = [
    ('Age', 'Customer Age Distribution', 'Age (Years)', '#3b82f6'),
    ('Annual Income (k$)', 'Annual Income Distribution', 'Income (k$)', '#10b981'),
    ('Spending Score (1-100)', 'Spending Score Distribution', 'Score (1-100)', '#8b5cf6')
]

for ax, (col, title, xlabel, color) in zip(axes, cols_meta):
    sns.histplot(df[col], kde=True, ax=ax, color=color, bins=15, edgecolor='black', alpha=0.6)
    mean_val = df[col].mean()
    median_val = df[col].median()
    ax.axvline(mean_val, color='#ef4444', linestyle='--', linewidth=1.5, label=f'Mean ({mean_val:.1f})')
    ax.axvline(median_val, color='#f59e0b', linestyle=':', linewidth=1.5, label=f'Median ({median_val:.1f})')
    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Number of Customers')
    ax.legend()

plt.tight_layout()
plt.show()


### Income vs. Spending Score Relationship
A scatter plot of Annual Income versus Spending Score reveals natural spatial groupings of customers.


In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df,
    x='Annual Income (k$)',
    y='Spending Score (1-100)',
    hue='Gender',
    palette={'Male': '#2563eb', 'Female': '#ec4899'},
    s=80,
    alpha=0.85,
    edgecolor='black',
    linewidth=0.5
)
plt.title('Customer Annual Income vs. Spending Score by Gender', fontsize=13, fontweight='bold')
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


### Correlation Heatmap of Numerical Features


In [ ]:
plt.figure(figsize=(7, 5))
corr_cols = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)', 'Gender_Encoded']
corr = df[corr_cols].corr()

sns.heatmap(
    corr,
    annot=True,
    cmap='coolwarm',
    fmt='.3f',
    vmin=-0.5,
    vmax=0.5,
    linewidths=1,
    cbar_kws={'label': 'Pearson Correlation'}
)
plt.title('Correlation Heatmap', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


**Key Insights from EDA**:
1. **Age vs Spending**: Age exhibits a noticeable negative correlation ($-0.327$) with Spending Score, demonstrating that younger demographics spend more actively in this mall.
2. **Income vs Spending**: There is virtually no linear correlation between Annual Income and Spending Score ($r = 0.0099$). However, the scatter plot clearly exhibits 5 distinct geometric clusters (four corners and one dense central cluster).
3. **Gender Distribution**: Female shoppers constitute 56% of mall visitors, while Male shoppers represent 44%.


## 5. K-Means Clustering
K-Means is an unsupervised clustering algorithm that groups $N$ observations into $K$ clusters by minimizing the within-cluster sum-of-squares (WCSS or inertia):

$$WCSS = \sum_{i=1}^{K} \sum_{x \in C_i} ||x - \mu_i||^2$$

where $\mu_i$ is the centroid of cluster $C_i$.

### Step 5.1: Feature Selection & Standardization
Because distance metrics in K-Means (Euclidean distance) are sensitive to feature scales, we standardize `Annual Income (k$)` and `Spending Score (1-100)` to zero mean and unit variance using `StandardScaler`.


In [ ]:
# Select features
features = ['Annual Income (k$)', 'Spending Score (1-100)']
X = df[features].values

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Feature matrix shape: {X_scaled.shape}")
print(f"Mean after scaling: {X_scaled.mean(axis=0).round(4)}")
print(f"Std dev after scaling: {X_scaled.std(axis=0).round(4)}")


### Step 5.2: Finding Optimal K via Elbow Method & Silhouette Analysis
We iterate $k$ from 1 to 10 and compute:
1. **Inertia (WCSS)**: The point where adding another cluster yields diminishing reduction in inertia represents the 'elbow'.
2. **Silhouette Coefficient**: Measures how similar an object is to its own cluster compared to neighboring clusters (ranging from -1 to +1). Peak score indicates the most cohesive and well-separated clusters.


In [ ]:
wcss = []
silhouette_vals = []
k_range = range(1, 11)

for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    km.fit(X_scaled)
    wcss.append(km.inertia_)
    if k >= 2:
        sil = silhouette_score(X_scaled, km.labels_)
        silhouette_vals.append(sil)
        print(f"k={k:2d} | WCSS = {km.inertia_:7.2f} | Silhouette Score = {sil:.4f}")
    else:
        print(f"k={k:2d} | WCSS = {km.inertia_:7.2f} | Silhouette Score = N/A")

# Plot Elbow and Silhouette Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Elbow plot
ax1.plot(list(k_range), wcss, marker='o', color='#2563eb', linewidth=2.5, markersize=8)
ax1.axvline(x=5, color='#ef4444', linestyle='--', linewidth=1.5, label='Elbow Point (k=5)')
ax1.set_title('The Elbow Method for Optimal k', fontsize=12, fontweight='bold')
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Inertia (WCSS)')
ax1.set_xticks(list(k_range))
ax1.legend()
ax1.grid(True, linestyle='--', alpha=0.5)

# Silhouette plot
ax2.plot(list(range(2, 11)), silhouette_vals, marker='s', color='#10b981', linewidth=2.5, markersize=8)
ax2.axvline(x=5, color='#ef4444', linestyle='--', linewidth=1.5, label='Peak Silhouette (k=5)')
ax2.set_title('Silhouette Score Analysis', fontsize=12, fontweight='bold')
ax2.set_xlabel('Number of Clusters (k)')
ax2.set_ylabel('Silhouette Coefficient')
ax2.set_xticks(list(range(2, 11)))
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


**Conclusion on Model Selection**:
- The Elbow curve exhibits a distinct inflection at **$k = 5$**.
- The Silhouette Coefficient reaches its global maximum of **$0.5547$** at **$k = 5$**.
- Therefore, $k = 5$ is both mathematically optimal and business-interpretable.


### Step 5.3: Training K-Means with Optimal $k=5$


In [ ]:
# Train final model
kmeans = KMeans(n_clusters=5, init='k-means++', random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

# Compute centroids in original feature units ($k and score)
centroids_scaled = kmeans.cluster_centers_
centroids_orig = scaler.inverse_transform(centroids_scaled)

print("K-Means training complete. Cluster assignments added to dataframe.")


### Step 5.4: Dynamic Segment Characterization & Business Profiling
Rather than assuming arbitrary cluster IDs, we inspect the exact centroid coordinates and dynamically categorize each segment.


In [ ]:
segment_profiles = {}

for i in range(5):
    inc = centroids_orig[i][0]
    spn = centroids_orig[i][1]
    cnt = int((df['Cluster'] == i).sum())
    pct = (cnt / len(df)) * 100
    
    if inc > 70 and spn > 60:
        name = "High Income, High Spending"
        tag = "VIP / Elite Shoppers"
        desc = "Affluent shoppers with high spending. Prime targets for luxury brands and VIP programs."
        strat = "Promote VIP loyalty tiers, concierge services, and exclusive private showcases."
    elif inc > 70 and spn <= 60:
        name = "High Income, Low Spending"
        tag = "Careful / Conservative Wealth"
        desc = "High earners who budget carefully and spend selectively."
        strat = "Focus on product quality, durability, and smart investment value."
    elif inc <= 40 and spn > 60:
        name = "Low Income, High Spending"
        tag = "Impulsive / Trendsetters"
        desc = "Lower income shoppers who spend eagerly on lifestyle and trendy items."
        strat = "Engage with flash promotions, BNPL payment options, and influencer collaborations."
    elif inc <= 40 and spn <= 60:
        name = "Low Income, Low Spending"
        tag = "Sensible / Budget Conscious"
        desc = "Frugal shoppers focused on essential purchases and strict budgets."
        strat = "Offer everyday discounts, bundle deals, and loyalty points for essentials."
    else:
        name = "Average Income, Average Spending"
        tag = "Core Mainstream / Balanced"
        desc = "The stable backbone of mall foot traffic with balanced earning and spending."
        strat = "Maintain recurring seasonal sales, rewards for repeat visits, and family events."
        
    segment_profiles[i] = {
        'name': name,
        'tag': tag,
        'centroid_income': round(inc, 1),
        'centroid_spending': round(spn, 1),
        'count': cnt,
        'percentage': round(pct, 1),
        'description': desc,
        'strategy': strat
    }

df['Segment_Name'] = df['Cluster'].map(lambda c: segment_profiles[c]['name'])
df['Segment_Tag'] = df['Cluster'].map(lambda c: segment_profiles[c]['tag'])

summary_df = pd.DataFrame.from_dict(segment_profiles, orient='index')
summary_df[['tag', 'centroid_income', 'centroid_spending', 'count', 'percentage']]


### Step 5.5: Customer Segmentation Cluster Visualization


In [ ]:
plt.figure(figsize=(11, 7))
cluster_colors = ['#10b981', '#8b5cf6', '#ec4899', '#3b82f6', '#64748b']

for i in range(5):
    subset = df[df['Cluster'] == i]
    plt.scatter(
        subset['Annual Income (k$)'],
        subset['Spending Score (1-100)'],
        s=75,
        color=cluster_colors[i],
        alpha=0.85,
        edgecolors='black',
        linewidths=0.5,
        label=f"{segment_profiles[i]['tag']} ({len(subset)})"
    )

# Centroids
plt.scatter(
    centroids_orig[:, 0],
    centroids_orig[:, 1],
    s=260,
    color='#dc2626',
    marker='*',
    edgecolors='black',
    linewidths=1.5,
    label='Cluster Centroids',
    zorder=10
)

for i in range(5):
    plt.annotate(
        f"C{i} (${centroids_orig[i, 0]:.0f}k, {centroids_orig[i, 1]:.0f})",
        (centroids_orig[i, 0], centroids_orig[i, 1]),
        textcoords="offset points",
        xytext=(0, 12),
        ha='center',
        fontsize=9,
        fontweight='bold',
        bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="#dc2626", alpha=0.85)
    )

plt.title('Mall Customer Segments Identified by K-Means (k=5)', fontsize=14, fontweight='bold', pad=12)
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True)
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


## 6. Supervised Customer Classification (Logistic Regression)

### Objective & Methodology
While K-Means groups existing shoppers into natural geometric clusters, mall operators often want to predict:
> *"Given a prospective customer's demographic profile (Age, Gender, Annual Income), will they have a **High Spending Propensity**?"*

**Methodology Compliance**:
- As stipulated in the guidelines, we **do not** use arbitrary unsupervised K-Means cluster IDs as ground-truth labels.
- Instead, we formulate a clear, real-world business target:
  - **High Spender ($y = 1$)**: $\text{Spending Score} \ge 50$
  - **Moderate/Low Spender ($y = 0$)**: $\text{Spending Score} < 50$
- We partition the dataset into an **80% training set** and **20% testing set** using stratified sampling.
- Preprocessing pipelines (scaling) are fit **only on training data** to prevent data leakage.


In [ ]:
# Define supervised target
df_clf = df.copy()
df_clf['High_Spender'] = (df_clf['Spending Score (1-100)'] >= 50).astype(int)

clf_features = ['Age', 'Annual Income (k$)', 'Gender_Encoded']
X_clf = df_clf[clf_features]
y_clf = df_clf['High_Spender']

print(f"Classification Target Distribution:")
print(y_clf.value_counts(normalize=True).round(3) * 100)

# Train/Test Split (80/20 Stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X_clf, y_clf, test_size=0.20, random_state=42, stratify=y_clf
)

# Fit scaler strictly on training data
log_scaler = StandardScaler()
X_train_scaled = log_scaler.fit_transform(X_train)
X_test_scaled = log_scaler.transform(X_test)

# Train Logistic Regression
log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_train_scaled, y_train)

# Predictions & Probabilities
y_pred = log_reg.predict(X_test_scaled)
y_prob = log_reg.predict_proba(X_test_scaled)[:, 1]

# Evaluation Metrics
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("--- Logistic Regression Test Results ---")
print(f"Accuracy:  {acc:.4f} ({acc*100:.1f}%)")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=['Mod/Low Spender', 'High Spender']))


### Confusion Matrix Analysis


In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Pred Low/Mod', 'Pred High Spender'],
    yticklabels=['Actual Low/Mod', 'Actual High Spender'],
    cbar=False
)
plt.title('Logistic Regression Confusion Matrix', fontsize=12, fontweight='bold', pad=10)
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()


### Interpretation of Logistic Regression Coefficients
Analyzing model weights provides key business interpretability:


In [ ]:
coef_table = pd.DataFrame({
    'Feature': clf_features,
    'Coefficient': log_reg.coef_[0].round(4),
    'Odds Ratio': np.exp(log_reg.coef_[0]).round(4)
})
print("Feature Coefficients & Odds Ratios:")
print(coef_table.sort_values(by='Coefficient', ascending=False))


**Key Classification Findings**:
1. **Age Effect**: `Age` has the strongest negative coefficient ($-0.5911$, Odds Ratio $\approx 0.55$). Each standard-deviation increase in age significantly decreases the odds of being a high spender in this mall environment.
2. **Income Effect**: `Annual Income` shows near-zero direct correlation with spending propensity ($b = -0.012$), echoing our EDA finding that wealth alone does not guarantee high mall spending.
3. **Complementary Roles**: K-Means groups existing shoppers for portfolio segmentation, whereas Logistic Regression enables proactive scoring of new prospects.


## 7. Model Serialization & Saving with Joblib
To allow the Flask web application to perform instant real-time predictions without retraining, we save the trained models, fitted scalers, and segment metadata.


In [ ]:
os.makedirs('../models', exist_ok=True)

# Save models and scalers
joblib.dump(kmeans, '../models/kmeans_model.joblib')
joblib.dump(scaler, '../models/scaler.joblib')
joblib.dump(log_reg, '../models/logistic_model.joblib')
joblib.dump(log_scaler, '../models/logistic_scaler.joblib')

# Save metadata
metadata = {
    'kmeans_features': features,
    'n_clusters': 5,
    'segments': segment_profiles,
    'dataset_summary': {
        'total_customers': len(df),
        'avg_income': round(float(df['Annual Income (k$)'].mean()), 1),
        'avg_spending': round(float(df['Spending Score (1-100)'].mean()), 1),
        'avg_age': round(float(df['Age'].mean()), 1)
    }
}
joblib.dump(metadata, '../models/segment_metadata.joblib')

print("All models, scalers, and metadata successfully serialized with Joblib.")


## 8. Summary & Conclusion

### Summary of Accomplishments
1. **Data Preprocessing**: Inspected 200 customer records, verified absence of nulls/duplicates, preserved legitimate high-income records ($137k) via IQR analysis, and standardized features.
2. **Exploratory Data Analysis**: Identified key demographic trends, including a strong inverse relationship between age and spending score.
3. **Unsupervised Customer Segmentation**: Evaluated cluster structures using the Elbow Method and Silhouette Analysis, establishing $k=5$ as the optimal grouping.
4. **Actionable Segment Profiles**: Categorized the customer base into 5 strategic segments:
   - *Core Mainstream* (40.5%): Balanced income and spending.
   - *VIP / Elite Shoppers* (19.5%): High income, high spending.
   - *Careful / Conservative Wealth* (17.5%): High income, low spending.
   - *Sensible / Budget Conscious* (11.5%): Low income, low spending.
   - *Impulsive / Trendsetters* (11.0%): Low income, high spending.
5. **Supervised Classification**: Successfully evaluated a Logistic Regression classifier achieving **72.5% accuracy** in predicting high-spending propensity.
6. **Production Readiness**: Serialized all pipelines for deployment in the Flask web dashboard.
